In [ ]:
!uv pip install -U langchain langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv


In [ ]:
load_dotenv(override=True)

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True,
    provider="auto"
)

model = ChatHuggingFace(llm=llm)


In [ ]:
response = model.invoke("How are you?")

print(response.content)

## Data Ingestion


In [ ]:
!uv pip install langchain_community

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "../data/Agentic AI.txt",
    encoding="utf-8"
)

documents = loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
 

In [ ]:
splitter= RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)


In [ ]:
chunks=splitter.split_documents(documents)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
import sys

!uv pip install --python "{sys.executable}" faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)

In [ ]:
retriever=vector_store.as_retriever(
    search_kwargs={'k':3}
)

In [ ]:
docs=retriever.invoke(
    "Who is Uzair"
)

In [ ]:
for i ,  doc in enumerate(docs):

    print(f"Number {i} :- {doc.page_content} \n")

In [39]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:
""",
    input_variables=["question", "context"]
)

In [38]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [37]:
from langchain_core.runnables import RunnablePassthrough


chain= ({"context":retriever ,"question":RunnablePassthrough()}|prompt | model | parser)

In [40]:
result =chain.invoke("Explain Agentic Ai")
result

'Agentic AI is designed to take actions and work towards specific goals, going beyond traditional AI which typically responds to user prompts. It analyzes a goal, breaks it into smaller tasks, decides on necessary actions, uses available tools, checks results, and continues until the task is completed. A key feature is its ability to plan complex tasks by dividing them into manageable steps. For example, preparing a report involves searching for information, collecting sources, organizing, writing, and reviewing. Another significant aspect is tool calling, allowing the AI to use external functions, such as weather tools or databases, to gather necessary information and complete tasks accurately.'